In [7]:
# add parent to path
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().resolve().parent.parent))

from plots.wandb_utils import combine_histories, get_wandb_stats

run_ids = [
    # 3/16/2026
    # "szm0buc8",  # storage
    "osfnlgy4",  # base
    # "ci0xo4tp",  # optim
]


hists_list = []
for run_id in run_ids:
    summary, hist, config = get_wandb_stats(
        run_id,
        skip_cache=True,  # set to True to skip cache and fetch from W&B API
        wandb_run_cache_path=Path("/mnt/labstore/bespoke_olap/wandb_cache"),
    )
    hists_list.append(hist)

hist = combine_histories(hists_list)

✓ Run loaded: tpch_initial1-22v2017_wstorage
  State: finished
  Created: 2026-03-16T16:08:13Z
✓ Data fetched: 1249 turns, 148 columns
✓ W&B data cached to: /mnt/labstore/bespoke_olap/wandb_cache/5b33965f31e508100cb883ffc844e186ae9f2dd02e62076e21abde65bd25dc6a.pkl
Combined history has 1249 rows ([1249])


In [8]:
# sort by _step
hist = hist.sort_values("_step")

# fill all following missing cell values
hist["current_prompt_descriptor"] = hist["current_prompt_descriptor"].ffill()

In [ ]:
import pandas as pd
from IPython.display import display
from plots.utils.annotate_speedup_col import annotate_total_speedup_per_turn

# Build a per-prompt summary table
rows = []
grouped = hist.groupby("current_prompt_descriptor", sort=False)

speedup_ls = []

for name, group in grouped:
    llm_calls = group[
        (group["type"] == "llm_call") & (group["supervisor"].isna())
    ].shape[0]

    supervisor_calls = group[group["supervisor"] == True].shape[0]
    supervisor_approvals = group[group["supervisor/approved"] == True].shape[0]
    interventions = supervisor_calls - supervisor_approvals
    intervention_rate = (interventions / supervisor_calls) if supervisor_calls else 0.0

    # extract step
    begin_step = group["_step"].iloc[0]
    end_step = group["_step"].iloc[-1]

    type_counts = {
        t: group[group["type"] == t].shape[0]
        for t in ["apply_patch", "validate", "shell", "compile"]
    }

    # extract first llm_hash in the group for reference
    llm_hash = group[group["type"] == "llm"]["llm_hash"].iloc[0]

    # compute speedup up to this point (like a prefix sum)
    hist_up_to_point = hist[hist["_step"] <= end_step]
    speedup_col_name = annotate_total_speedup_per_turn(hist_up_to_point)

    # extract last speedup val
    val_rows = hist_up_to_point[
        (hist_up_to_point["type"] == "validate")
        & (hist_up_to_point["validation/trace_mode"] == False)
        & (hist_up_to_point["validation/compile_with_optimize"] == True)
    ]
    speedup_vals = val_rows[speedup_col_name].dropna()
    if len(speedup_vals) > 0:
        speedup = (
            speedup_vals.iloc[-1] if speedup_col_name in hist_up_to_point else None
        )
    else:
        speedup = None
        speedup_ls.append(hist_up_to_point)

    rows.append(
        {
            "step": begin_step,
            "prompt": name,
            "llm_calls": llm_calls,
            "supervisor_calls": supervisor_calls,
            "approvals": supervisor_approvals,
            "interventions": interventions,
            "intervention_rate": intervention_rate,
            **type_counts,
            "hash of first llm call": llm_hash,
            "speedup": speedup,
        }
    )

summary_df = pd.DataFrame(rows)

# Make numeric columns explicit for stable styling
for col in [
    "llm_calls",
    "supervisor_calls",
    "approvals",
    "interventions",
    "intervention_rate",
    "apply_patch_tool",
    "validate",
    "shell_command",
    "compile",
]:
    if col in summary_df.columns:
        summary_df[col] = pd.to_numeric(summary_df[col], errors="coerce")

# Visual styling: higher interventions => red, lower interventions => green
styled = (
    summary_df.style.format({"intervention_rate": "{:.1%}"})
    .set_caption("Supervisor and Tool Activity by Prompt")
    .background_gradient(cmap="RdYlGn_r", subset=["interventions", "intervention_rate"])
    # .background_gradient(cmap="RdYlGn", subset=["approvals"])
    .background_gradient(cmap="Blues", subset=["llm_calls", "supervisor_calls"])
    .bar(
        color="#3B82F6",
        subset=["apply_patch", "validate", "shell", "compile"],
    )
)

display(styled)

,step,prompt,llm_calls,supervisor_calls,approvals,interventions,intervention_rate,apply_patch,validate,shell,compile,hash of first llm call,speedup
0,0,base impl planner,0,1,1,0,0.0%,0,0,3,0,943ec6965de97fd253814bd45e6bf040d63872b827c1b50963fd5a3aa67470f5,nan
1,8,base impl storage,0,1,1,0,0.0%,3,1,6,1,e219672477bc78a8a528f591d77cbd55666df69467ba58c67989b599c1729b9d,nan
2,30,optimize build,0,5,1,4,80.0%,9,13,35,9,5eb98231ea7d3d9911e7224a4d83348a0df26b62ac4cfb34c69607d9e3dbe525,1.678637
3,84,add time measurement to query execution,0,2,1,1,50.0%,2,0,4,2,bec707f795537ece53f7cee672ffd83cd1ed3b06b7df7c7cfb5e648057786727,nan
4,105,base impl Q1,0,2,1,1,50.0%,5,0,17,2,9e1b715bc52bfd7b04b58f55f1b16e88ff86c5ead56eaf11fe35929233e785ae,nan
5,154,base impl exec & validate Q1,0,1,1,0,0.0%,4,7,7,3,1341b50e3f54dd634152293279d499950a4838a99721d71466f87fee47af1612,7.438523
6,197,base impl Q2,0,1,1,0,0.0%,5,4,13,3,5117b83c0abeb9a152a3975f040555d9a9e24b8a52eff7a08e7e4cd239227feb,nan
7,247,base impl exec & validate Q2,0,1,1,0,0.0%,0,4,0,0,f5d0158e0bb3bcbd7552606a61139858611f3eff6f63997b90493cf36850ead6,7.478139
8,256,base impl Q3,0,1,1,0,0.0%,5,0,7,1,46a01aeb134b6c678f85913e5c38395d2f13eda6a310cc85ca1cedac62d9238b,7.478139
9,282,base impl exec & validate Q3,0,1,1,0,0.0%,5,8,1,4,861f399d0bbbc5ffd82a81a5e973fd889682896c69a9c93418fbdf0b1ed63688,1.864804


In [10]:
test_hist = hist[hist["_step"] <= 726]

In [11]:
col_name = annotate_total_speedup_per_turn(test_hist)

data = test_hist[col_name].dropna()
print(data)

724    1.874229
725    1.874229
726    1.874229
Name: validation/largest_sf_total_speedup, dtype: float64


In [12]:
# analyze activities after step
start_step = 0  # first step of last prompt, can adjust as needed
start_step = summary_df["step"].iloc[-1]  # start of last prompt

filtered_hist = hist[hist["_step"] >= start_step]

summary_rows = []
grouped = filtered_hist.groupby("current_prompt_descriptor", sort=False)

for name, group in grouped:
    # iterate over each row in the group
    summary_rows.append(f"=== Prompt: {name} ===")
    for _, row in group.iterrows():
        step = row["_step"]
        if row["type"] in ["validate"]:
            if "validation/correct" not in row:
                correct_str = "unknown"
            else:
                correct_str = (
                    "correct✅" if row["validation/correct"] else "incorrect❌"
                )
            summary_rows.append(
                f"[{step}] - Validate: {correct_str} / SF={row['validation/scale_factor']} / Tracing={row['validation/trace_mode']}"
            )
        elif row["type"] in ["apply_patch"]:
            summary_rows.append(f"[{step}] - Apply Patch")
        elif row["type"] in ["shell"]:
            summary_rows.append(f"[{step}] - Shell Command: {row['shell/commands']}")
        elif row["type"] in ["compile"]:
            summary_rows.append(
                f"[{step}] - Compile: {'success' if not row['compile/error'] else 'failure❌'}"
            )
        elif row["type"] == "llm" and row["supervisor"] != True:
            summary_rows.append(f"[{step}] - LLM Call")
        elif row["type"] == "llm" and row["supervisor"] == True:
            summary_rows.append(
                f"[{step}] - Supervisor LLM Call: {'approved✅' if row['supervisor/approved'] else 'rejected❌'}"
            )
        elif row["type"] == "compaction":
            summary_rows.append(f"[{step}] - Compaction Event")
        else:
            raise ValueError(f"Unexpected row type: {row['type']}")

for row in summary_rows:
    print(row)

=== Prompt: run all queries and fix any errors ===
[1134] - LLM Call
[1135] - Validate: correct✅ / SF=20.0 / Tracing=False
[1136] - LLM Call
[1137] - Supervisor LLM Call: approved✅
[1138] - Validate: correct✅ / SF=20.0 / Tracing=False
[1139] - Validate: correct✅ / SF=20.0 / Tracing=False
[1140] - Validate: correct✅ / SF=20.0 / Tracing=False
[1141] - Validate: correct✅ / SF=20.0 / Tracing=False
[1142] - Validate: correct✅ / SF=20.0 / Tracing=False
[1143] - Validate: correct✅ / SF=20.0 / Tracing=False
[1144] - Validate: correct✅ / SF=20.0 / Tracing=False
[1145] - Validate: correct✅ / SF=20.0 / Tracing=False
[1146] - Validate: correct✅ / SF=20.0 / Tracing=False
[1147] - Validate: correct✅ / SF=20.0 / Tracing=False
[1148] - Validate: correct✅ / SF=20.0 / Tracing=False
[1149] - Validate: correct✅ / SF=20.0 / Tracing=False
[1150] - Validate: correct✅ / SF=20.0 / Tracing=False
[1151] - Validate: correct✅ / SF=20.0 / Tracing=False
[1152] - Validate: correct✅ / SF=20.0 / Tracing=False
[1153] - 